# N056 · 单调队列与带负数的最短区间

**目标：** 同时维护候选顺序和过期边界。

**先修：** N032, N053, N054。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 滑窗极值；前缀和支配；双端弹出；负数使普通窗口失效。

**配套讲解来源：** [同名逐章意见](../../comment/056_monotonic_queue_windows.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章的主角是**单调队列**：它和单调栈思想一样（维持单调性、弹掉没用的候选），但两头都能弹——因为窗口问题里的候选不仅要"比大小"，还要"看保质期"。两个代表题：

1. **滑动窗口最大值**：一个长度为 k 的窗口从左往右滑，每一步报告窗口里的最大值。输入 `[1,3,-1,-3,5,3,6,7]`，k=3 → 输出 `[3,3,5,5,6,7]`。为什么第一个是 3：第一个窗口 `[1,3,-1]` 里 3 最大；窗口滑到 `[3,-1,-3]` 时 3 还在，仍报告 3；再滑一步 3 出窗，窗口 `[-1,-3,5]` 的最大值变成 5。暴力每步扫 k 个数是 O(nk)；单调队列让每个下标进出各一次，O(n)。
2. **和至少为 K 的最短子数组（含负数）**：找一段连续子数组，元素和 ≥ k，问最短长度。输入 `[2,-1,2]`，k=3 → 输出 3。为什么：任意更短的子数组（比如 `[2]`、`[2,-1]`、`[-1,2]`）的和都不到 3，只有整个数组 `[2,-1,2]` 的和恰好是 3。**负数是大麻烦**：N028/N029 那套"和不够就右扩、和够了就左缩"的普通窗口在这里会失效（子数组和随扩张不再是只增不减），所以要换前缀和 + 单调队列的思路。

两题的暴力都是枚举所有区间再算和/最值，O(n²) 起步。n=10⁵ 时约百亿次数组访问，跑不动；而单调队列的每个下标至多入队一次、出队一次（不管从哪头出），总量 O(n)。本章的目标（“学习目标”部分 原话）是"同时维护候选顺序和过期边界"——普通单调栈只会扔掉"弱的"，单调队列还得扔掉"老的"，两头各司其职。

## 2. 基础知识：先读懂这些词

- **单调队列（monotonic queue）**：用双端队列（deque，N053 学过：两头都能 O(1) 进出）维护的候选序列，从队首到队尾保持单调（本章窗口题是"对应的值递减"）。队首永远是目前的最优候选。
- **滑窗极值**：滑动窗口最值问题的简称。队列里存**下标**，因为"什么时候过期"（滑出窗口左界）只有下标能判断。
- **过期边界**：窗口每右移一格，最老的元素可能已经不在 `[i-k+1, i]` 范围里。队首下标一旦 `<= i-k` 就过期，从**前端**弹出——这是栈做不到、必须用双端队列的原因。
- **支配（dominance）**：“思路推导”部分 的原话"一个更早且不更大的值可永久删除，因为它先过期且不占优势"。翻译：老候选 j 比新来的 x 出现早（会更早滑出窗口）、值又不比 x 大，那 j 在任何未来窗口里都不可能是最大值——x 在的一天 j 被 x 压着，x 走了 j 早就走了。这种"又老又弱"的候选从**后端**弹出。
- **前缀和（prefix sum）**：`prefix[i]` = 前 i 个元素之和，子数组 `a[l..r) `的和就是 `prefix[r]-prefix[l]`（N031 学过）。它把"找和 ≥ k 的区间"变成"找一对前缀，差 ≥ k"。
- **前缀支配**：最短区间题里的对应概念——若下标 b 比 a 晚（b>a）且 `prefix[b] <= prefix[a]`，那么 b 全面碾压 a：以 b 为左端，任何右端 i 的区间和 `prefix[i]-prefix[b] >= prefix[i]-prefix[a]`（更容易达标），而且区间更短。a 从此没用了，从队尾弹掉。
- **递增前缀队列**：弹完被支配者后，队列里的前缀值从队首到队尾严格递增。队首是"最小、最老"的前缀——最容易让差达到 k 的左端候选。
- **负数使普通窗口失效**：有负数时，窗口左缩（丢掉一个负数）反而可能让和变大、右扩（加一个负数）反而让和变小，"和太小就右扩"的贪心前提（单调性）不复存在。

## 3. 思路推导：从小例子开始

**max_sliding_window 的步骤（每个 i 做三件事）：**

1. **过期清理**：队首下标若 `<= i-k`，说明它已经滑出窗口，从队首弹出。
2. **弱者清理**：只要队尾下标对应的值 `<= x`（新元素），从队尾弹出——它们比 x 老、又不比 x 大，永无出头之日（支配关系）。
3. x 的下标入队尾；若窗口已满（`i>=k-1`），队首下标对应的值就是本窗口最大值，记录。

**手算演示：`[1,3,-1,-3,5,3,6,7]`，k=3——**

| i | 新值 | 过期弹出 | 队尾弹出 | 队列（下标） | 输出 nums[队首] |
|---|------|-----------|-----------|----------------|------------------|
| 0 | 1 | — | — | [0] | （窗口未满） |
| 1 | 3 | — | 0（值1≤3） | [1] | （窗口未满） |
| 2 | -1 | — | — | [1,2] | 3 |
| 3 | -3 | — | — | [1,2,3] | 3 |
| 4 | 5 | 1（1≤4-3） | 3、2（都≤5） | [4] | 5 |
| 5 | 3 | — | — | [4,5] | 5 |
| 6 | 6 | — | 5、4（3≤6、5≤6） | [6] | 6 |
| 7 | 7 | — | 6（6≤7） | [7] | 7 |

重点看 i=4：过期清理先弹掉下标 1（值 3 出窗），然后 5 一个接一个把 -3、-1 都从队尾清掉——队列从 `[1,2,3]` 一步变成 `[4]`，输出直接跳到 5。

**shortest_subarray 的步骤：**

1. 先算前缀和数组 `prefix`（长度 n+1，`prefix[0]=0` 代表空前缀）。
2. 从左到右枚举右端 i 的前缀 `value`，对递增队列做两种弹出：
   - **队首弹出（可行即结算）**：`value - prefix[队首] >= k` 说明从队首下标到 i 的区间和已达标，长度 `i-队首` 记入候选，然后弹掉它——为什么敢弹？因为队首是最老的候选，i 再往后走区间只会更长，它的历史使命已经完成。
   - **队尾弹出（被支配）**：`prefix[队尾] >= value` 说明队尾这个左端比 i 晚还比 i 大（前缀值大意味着后续区间和更难达标），被 i 全面支配，弹掉。
   - i 入队尾。
3. 全程最优长度与 `len(nums)+1`（哨兵"无解"值）比较，仍无解则返回 -1。

**手算演示（“运行示例”部分 的表格，a=[2,-1,2]，k=3，prefix=[0,2,1,3]）：**

| i | P[i] | 本次可行区间 | 弹队尾 | 队列（候选左前缀） |
|---|------|----------------|---------|----------------------|
| 0 | 0 | — | — | [0] |
| 1 | 2 | —（2-0=2<3） | —（P[0]=0<2） | [0, 1] |
| 2 | 1 | —（1-0=1<3） | 1（P[1]=2≥1，被支配） | [0, 2] |
| 3 | 3 | (0,3)，长度 3-0=3 | —（P[2]=1<3） | [2, 3] |

i=2 那行是精髓：前缀 1 把前缀 2 支配掉（更晚出现、前缀更小），所以下标 1 永久出局；到 i=3 时用队首的 0 一算差值 3≥3，得到长度 3 的可行区间，同时把 0 结算弹掉。最终答案 3。

**再演示一次"连续结算"**：设某数组的前缀是 `prefix=[0,1,5,9]`，k=4。i=0、i=1 无事发生，队列为 [0,1]；到 i=2（值 5）时精彩了：5−0=5≥4，结算长度 2−0=2，弹出 0；新队首 1：5−1=4≥4 仍达标，结算更短的长度 2−1=1，再弹出；队列空了才停，随后 2 入队。一个 i 连续结算两次，最短立刻从 2 更新到 1。这说明队首弹出必须用 **while** 不能用 if——弹掉一个队首后，下一个队首的差可能仍然 ≥ k，而它给出的区间更短，不继续试就会漏掉最优。

**普通双指针在这里怎么翻车**：拿 `[2,-1,2]` 想象贪心——右扩到整段和才到 3，记长度 3；左缩掉 2 后和剩 1，不达标，贪心回答 3，这题碰巧对。但换 `[1,-1,5]`、k=5：贪心一路右扩到整段 `[1,-1,5]` 和 5 达标，记长度 3；左缩掉 1 后和剩 4，不达标，回答 3——而正确答案是 `[5]` 长度 1。翻车原因：左缩掉的是正数 1，和变小了，指针误以为不能再缩；可实际上"缩到只剩 5"更短也达标，普通窗口的收缩规则根本探索不到这种形状。

## 4. 核心代码：max_sliding_window

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def max_sliding_window(nums, k):
    if not nums:
        return []
    if not 1 <= k <= len(nums):
        raise ValueError('invalid window size')
    queue = deque()
    answer = []
    for i, x in enumerate(nums):
        while queue and queue[0] <= i - k:
            queue.popleft()
        while queue and nums[queue[-1]] <= x:
            queue.pop()
        queue.append(i)
        if i >= k - 1:
            answer.append(nums[queue[0]])
    return answer
```

### 逐段读懂代码

**滑动窗口最大值（对应实现）：**

```python
def max_sliding_window(nums, k):
    if not nums:
        return []
    if not 1 <= k <= len(nums):
        raise ValueError('invalid window size')
    queue = deque()
    answer = []
    for i, x in enumerate(nums):
        while queue and queue[0] <= i - k:
            queue.popleft()
        while queue and nums[queue[-1]] <= x:
            queue.pop()
        queue.append(i)
        if i >= k - 1:
            answer.append(nums[queue[0]])
    return answer
```

- 前两行是输入契约的守门：空数组直接返回空结果；k 不在 `[1, len(nums)]` 里（0、负数或比数组还长）就抛异常，不给无意义答案。
- 第一个 while 是**过期清理**：`queue[0]<=i-k` 表示队首已经滑出窗口左边界（窗口是 `[i-k+1, i]`），用 `popleft()` 从前端弹。注意 `1<=k` 保证 `i-k` 单调不减，所以队首过期检查不会漏。
- 第二个 while 是**弱者清理**：`nums[queue[-1]]<=x` 用 `<=`——相等也弹。相等的旧值比新值先过期，留着没有任何好处（这和 N054 的"严格"不同，因为这里多了过期维度，相等即可支配）。
- `queue.append(i)` 把新下标压到队尾；此刻队列对应的值从队首到队尾严格递减。
- `if i>=k-1`：窗口第一次装满是 i=k-1 时，从这步起每个 i 都输出一次 `nums[queue[0]]`——队首存活者就是窗口最大值。

**最短区间（对应实现）：**

```python
def shortest_subarray(nums, k):
    prefix = [0]
    for x in nums:
        prefix.append(prefix[-1] + x)
    queue = deque()
    best = len(nums) + 1
    for i, value in enumerate(prefix):
        while queue and value - prefix[queue[0]] >= k:
            best = min(best, i - queue.popleft())
        while queue and prefix[queue[-1]] >= value:
            queue.pop()
        queue.append(i)
    return best if best <= len(nums) else -1
```

- 前两行递推构造前缀和，`prefix[0]=0` 让"从数组开头出发的区间"也有对应的左端点，不需要特判。
- `best=len(nums)+1` 是"还没找到"的哨兵值——任何真实区间长度最多是 len(nums)，比它大就意味着无解。
- 第一个 while 是**可行结算**：队首是最小、最老的前缀，只要 `value-队首前缀 >= k` 就得到一个长度 `i-queue.popleft()` 的可行区间；注意弹的动作在 min 的参数里完成（先取长度再弹出）。用 while 而不是 if：弹掉一个队首后，下一个队首的差可能仍 ≥ k，而它给出的区间更短，要继续试。
- 第二个 while 是**支配清理**：`prefix[queue[-1]]>=value`（大于等于）说明旧队尾被 i 支配——i 更晚（区间更短）且前缀不更大（差更大、更容易达标），旧队尾永无翻身之日。
- `return best if best<=len(nums) else -1`：找到了就返回最短长度，否则按题意返回 -1。

## 5. 分段实现：按顺序运行

**本章接口：** `max_sliding_window(nums, k)`、`shortest_subarray(nums, k)`

### 导入本章使用的库

In [1]:
from collections import deque

### max_sliding_window

In [2]:
def max_sliding_window(nums, k):
    if not nums:
        return []
    if not 1 <= k <= len(nums):
        raise ValueError('invalid window size')
    queue = deque()
    answer = []
    for i, x in enumerate(nums):
        while queue and queue[0] <= i - k:
            queue.popleft()
        while queue and nums[queue[-1]] <= x:
            queue.pop()
        queue.append(i)
        if i >= k - 1:
            answer.append(nums[queue[0]])
    return answer

### shortest_subarray

In [3]:
def shortest_subarray(nums, k):
    prefix = [0]
    for x in nums:
        prefix.append(prefix[-1] + x)
    queue = deque()
    best = len(nums) + 1
    for i, value in enumerate(prefix):
        while queue and value - prefix[queue[0]] >= k:
            best = min(best, i - queue.popleft())
        while queue and prefix[queue[-1]] >= value:
            queue.pop()
        queue.append(i)
    return best if best <= len(nums) else -1

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[2,-1,2]; p=[0]
for x in a: p.append(p[-1]+x)
q=deque(); rows=[]
for i,v in enumerate(p):
    solved=[]
    while q and v-p[q[0]]>=3: solved.append((q[0],i,i-q[0])); q.popleft()
    while q and p[q[-1]]>=v: q.pop()
    q.append(i); rows.append((i,v,list(q),solved))
show_table(['i','P[i]','候选左前缀','本次可行区间'],rows)

i,P[i],候选左前缀,本次可行区间
0,0,[0],[]
1,2,"[0, 1]",[]
2,1,"[0, 2]",[]
3,3,"[2, 3]","[(0, 3, 3)]"


## 7. 正确性、适用条件与复杂度

更晚且前缀更小的索引，对任意未来右端都给出不更难满足的和与更短长度，支配旧索引。队首一旦已可行，其最短可用右端已检查，未来只会更长。

**代价：** 两个算法 O(n)时间；窗口最大O(k)辅助空间，前缀最短区间O(n)。

### 展开理解

- **窗口最大值为什么对**：任何被队尾弹出的下标 j，都存在一个更晚出现且值不小于它的下标 i——i 在窗口里的每一天 j 也在（j 更早进窗就更早出窗），而 i 的值压着 j，所以 j 在它存活的任何窗口里都当不上最大值，删掉它不损失任何答案。队首过期弹出同理——它已经不在窗口里了。于是每一步留下的队首恰好是"还活着且没被支配"的最老最强者，就是最大值。
- **最短区间为什么对**：“正确性与复杂度”部分 的论证翻译过来有两半。其一（队尾弹出的正确性）：更晚且前缀更小的下标，对任何未来右端点给出的区间都"和不更难达标、长度还更短"，所以被支配的旧下标可以放心扔。其二（队首弹出的正确性）：当右端 i 走到队首已经可行时，i 就是队首能配到的**最短**右端（i 再往后走长度只增不减），此刻不结算以后就没机会了，结 完即可丢弃。两个方向的丢弃都不丢最优解，枚举又覆盖了所有"可行配对的最短代表"，所以最优长度一定被比较到。
- **为什么负数逼我们换方法**：普通双指针依赖"窗口右扩则和增、左缩则和减"。负数加进来后，左缩掉一个 -5 和反而暴涨，右扩加个 -3 和反而下跌，单调性崩塌，贪心收缩会错过"绕过负数段"的短答案。前缀差视角不看连续扩张，只看配对，天然不受影响。
- **复杂度**：两个算法里每个下标至多入队一次、出队一次（不管从前端还是后端），总操作是 O(n)。窗口最大值额外空间 O(k)（队列最多存一个窗口的下标）；最短区间要存前缀数组和队列，是 O(n)。n=10⁵ 时各是几十万次队操作，毫秒级完成；暴力 O(nk) 在 n=k=10⁵ 时是 10¹⁰ 次比较，根本不可行。
- **失效条件与输入契约**：`max_sliding_window` 明确要求 `1<=k<=len(nums)`，k 越界抛 ValueError——这不是啰嗦，而是把"窗口比数组长"这类无定义输入挡在门外；空数组则是合法输入，返回空列表。`shortest_subarray` 无解时按题意返回 -1，用 `len(nums)+1` 当"无穷大"哨兵而不是 float('inf')，全程整数运算更干净。

本章的踩坑清单，每条都对应第三节演示里的某个瞬间：

- 窗口题过期条件写成 `queue[0]<i-k`：少弹一格，恰好滑出边界的旧最大值会多报一步（正确是 `<=`）。
- 窗口题队尾清理写成 `<`：相等值留在队里白白占位，虽然答案仍对，但队列不再严格递减、浪费空间还容易想乱。
- 窗口题把 `if i>=k-1` 忘掉：前 k-1 步窗口未满也输出，答案多出一段。
- 最短区间忘建 `prefix[0]=0`：所有"从下标 0 出发"的区间被漏掉。
- 最短区间队首结算写成 if：`prefix=[0,1,5,9]`、k=4 的连续结算场景直接漏掉更短的 1。
- 前缀和数组算成 `prefix[i]=nums[i]`（不做累加）：所有差值全错，属于"公式背了、数组没建对"的典型。

## 8. 单元测试：每个用例在检查什么

- `assert max_sliding_window([1,3,-1,-3,5,3,6,7],3)==[3,3,5,5,6,7]`：LC239 的经典用例，一次覆盖三种事件——最大值存活多个窗口（3 连报两次）、过期清理（i=4 弹掉下标 1）、连续队尾清理（5 一口气清掉 -1、-3）。
- `assert shortest_subarray([2,-1,2],3)==3`：LC862 的经典用例。中间的 -1 让前缀先升后降（0→2→1→3），正好触发"队尾支配弹出"（i=2 弹掉前缀 2）和"队首可行结算"（i=3 配队首 0）两条关键路径。
- `assert shortest_subarray([1,2],9)==-1`：无解边界——任何子数组的和都到不了 9，best 保持哨兵值 len+1，必须返回 -1。
- 三层穷举（`n in range(6)`，值取 {-1,0,2}，共 1+3+9+27+81+243=364 个数组）：对每个数组，k 从 1 到 4 用双重循环暴力枚举所有子数组的 `sum(a[l:r])>=k` 对比最短长度；k 从 1 到 n 用 `max(a[i:i+k])` 对比窗口最大值。值域含负数 -1 和 0，专门让前缀和上蹿下跳、让相等值反复出现（窗口题用的是 `<=` 支配，相等值是否弹对也要靠穷举敲打）。小规模全枚举等于把实现和暴力互为照妖镜。
- 注意窗口最大值那部分穷举的 k 只取到 n——k 大于 n 会被实现抛 ValueError，穷举范围正好停在合法边界内；而最短区间的 k 取 1 到 4 故意超过 n（比如 n=2、k=9 的无解场景），就是要覆盖"怎么凑都差得远、必须返回 -1"的分支。两段穷举的取值范围本身就是输入契约的注释。

In [5]:
assert max_sliding_window([1, 3, -1, -3, 5, 3, 6, 7], 3) == [3, 3, 5, 5, 6, 7]

assert shortest_subarray([2, -1, 2], 3) == 3

assert shortest_subarray([1, 2], 9) == -1

from itertools import product

for n in range(6):
    for a in product([-1, 0, 2], repeat=n):
        for k in range(1, 5):
            lengths = [r - l for l in range(n) for r in range(l + 1, n + 1) if sum(a[l:r]) >= k]
            assert shortest_subarray(a, k) == (min(lengths) if lengths else -1)
        for k in range(1, n + 1):
            assert max_sliding_window(a, k) == [max(a[i:i + k]) for i in range(n - k + 1)]

print('N056: 所有本章断言通过')

N056: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 解释最短区间中两种弹出条件。

**练习 2：** 构造负数窗口反例。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（解释两种弹出条件）**：提示：把两个 while 各写一句话——队首弹出回答"谁已经够好了"（达到阈值就结算，而且必须现在结算，因为它的最短右端已经出现）；队尾弹出回答"谁永远好不了"（被更晚更小的前缀全面碾压）。再用手算把 `[2,-1,2], k=3` 的 i=2（队尾弹）和 i=3（队首弹）两步各画一张队列表格。追问自己一个边界：如果把队首弹出改成 `if`，哪个用例会漏？构造一个"同一 i 连续结算两个队首"的前缀序列（比如 prefix=[0,1,5,9] 配 k=4）。
- **练习 2（负数窗口反例）**：提示：目标是造一个数组，让"和不够就右扩、够了就左缩"的普通双指针给出错误答案。试 `[1,-1,5]` 配 k=5：双指针一路右扩到整段 `[1,-1,5]` 和才到 5，记下长度 3；左缩掉 1 后和剩 4 不达标，于是它回答 3——但正确答案是 `[5]` 长度 1。原因：左缩掉的是负数段前的正数，把和弄小了，指针逻辑误以为不能再缩。你把这个例子跑通暴力对照后，再用 `shortest_subarray([1,-1,5],5)==1` 验证单调队列版本是对的。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
from collections import deque

def max_sliding_window(nums, k):
    if not nums:
        return []
    if not 1 <= k <= len(nums):
        raise ValueError('invalid window size')
    queue = deque()
    answer = []
    for i, x in enumerate(nums):
        while queue and queue[0] <= i - k:
            queue.popleft()
        while queue and nums[queue[-1]] <= x:
            queue.pop()
        queue.append(i)
        if i >= k - 1:
            answer.append(nums[queue[0]])
    return answer

def shortest_subarray(nums, k):
    prefix = [0]
    for x in nums:
        prefix.append(prefix[-1] + x)
    queue = deque()
    best = len(nums) + 1
    for i, value in enumerate(prefix):
        while queue and value - prefix[queue[0]] >= k:
            best = min(best, i - queue.popleft())
        while queue and prefix[queue[-1]] >= value:
            queue.pop()
        queue.append(i)
    return best if best <= len(nums) else -1

# 示例与回归测试
assert max_sliding_window([1, 3, -1, -3, 5, 3, 6, 7], 3) == [3, 3, 5, 5, 6, 7]

assert shortest_subarray([2, -1, 2], 3) == 3

assert shortest_subarray([1, 2], 9) == -1

from itertools import product

for n in range(6):
    for a in product([-1, 0, 2], repeat=n):
        for k in range(1, 5):
            lengths = [r - l for l in range(n) for r in range(l + 1, n + 1) if sum(a[l:r]) >= k]
            assert shortest_subarray(a, k) == (min(lengths) if lengths else -1)
        for k in range(1, n + 1):
            assert max_sliding_window(a, k) == [max(a[i:i + k]) for i in range(n - k + 1)]

print('N056: 所有本章断言通过')

N056: 所有本章断言通过


## 11. 代表题与迁移

- **239. Sliding Window Maximum**：练"过期从队首弹、被支配从队尾弹"的单调双端队列，`max_sliding_window` 就是本题标准解。
- **862. Shortest Subarray with Sum at Least K**（[官方入口](https://leetcode.com/problems/shortest-subarray-with-sum-at-least-k/)）：练前缀和 + 递增前缀队列的两种弹出（可行结算与支配清理），以及为什么负数让普通滑窗失效，`shortest_subarray` 直接对应本题。

本章和 N054/N055 连成单调结构三部曲：栈处理"谁被谁支配"，贡献栈处理"谁负责多大范围"，队列再加上"谁先过期"。遇到新题先问一句——淘汰候选的理由是"更弱"、"更晚"还是"更老"，答案就指向对应的结构。

**题面入口：** [862](https://leetcode.com/problems/shortest-subarray-with-sum-at-least-k/)

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。